# Geo Migration — Country / Region / City / Locality

**Migration 1 of the run order** — run this before the user migration (and any content migration).

Migrates the legacy Strapi geo entities into the new 4-tier hierarchy.

**Read [docs/migrations/geo-migration.md](../docs/migrations/geo-migration.md) first** — full field mapping, what is dropped, and what needs manual work.

| Legacy (Strapi) | New (postcardv2) |
|---|---|
| `country` | `countries` |
| `region`  | `regions` |
| *(none — legacy has no City tier)* | `cities` — **synthesized**: one placeholder city per region |
| `locality` (hangs off Region) | `localities` (hangs off the region's placeholder city) |

**Prerequisites:** schema migrated (`npm run migrate:deploy`) and `python scripts/seed.py` run. Idempotent — safe to re-run.

In [20]:
import os, re
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=60)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])


connected to: postcardv2_test


## 1. Countries

Legacy `code`, `continent`, `otherNames`, `coverImage` have no column in the new schema — they are **dropped** here (see the mapping doc if they need preserving).

In [21]:
conn.rollback()  # clear any aborted transaction from a previous failed run

countries = fetch_all("/api/countries")
print(f"fetched {len(countries)} countries")

skipped = []
used_slugs = set()


def unique_slug(base):
    """Ensure slug uniqueness within this run (two names can slugify the same)."""
    base = base or "item"
    slug, n = base, 2
    while slug in used_slugs:
        slug = f"{base}-{n}"
        n += 1
    used_slugs.add(slug)
    return slug


with conn.cursor() as cur:
    for c in countries:
        a = attrs(c)
        name = (a.get("name") or "").strip()
        if not name:
            skipped.append(c["id"])
            continue
        slug = unique_slug(a.get("slug") or slugify(name))
        cur.execute(
            """
            INSERT INTO countries (name, slug) VALUES (%s, %s)
            ON CONFLICT (name) DO UPDATE SET slug = EXCLUDED.slug
            """,
            (name, slug),
        )
conn.commit()
print(f"upserted countries; skipped (no name): {skipped}")


fetched 268 countries
upserted countries; skipped (no name): []


## 2. Regions

Legacy Region has **no slug** (generated here) and is globally unique by name; new schema is unique per `(name, country_id)`. Regions without a parent country are collected for manual review.

In [16]:
conn.rollback()  # clear any aborted transaction from a previous failed run

regions = fetch_all("/api/regions", {"populate": "country"})
print(f"fetched {len(regions)} regions")

orphan_regions = []
with conn.cursor() as cur:
    for r in regions:
        a = attrs(r)
        name = (a.get("name") or "").strip()
        country = rel(a.get("country"))
        if not name or not country:
            orphan_regions.append((r["id"], name))
            continue
        cur.execute(
            """
            INSERT INTO regions (country_id, name, slug)
            SELECT id, %s, %s FROM countries WHERE name = %s
            ON CONFLICT (name, country_id) DO UPDATE SET slug = EXCLUDED.slug
            """,
            (name, slugify(name), (country.get("name") or "").strip()),
        )
conn.commit()
print(f"upserted regions; MANUAL REVIEW (no country): {orphan_regions}")


fetched 704 regions
upserted regions; MANUAL REVIEW (no country): [(254, 'Dominica')]


## 3. Cities — synthesized placeholder per region

The legacy system has **no City tier** (localities hang directly off regions). The new schema requires Country → Region → **City** → Locality, so we create one placeholder city per region carrying the region's own name.

⚠️ **MANUAL WORK**: rename/split these placeholder cities into real cities afterwards, and re-attach localities where a region actually spans several cities.

In [17]:
with conn.cursor() as cur:
    cur.execute(
        """
        INSERT INTO cities (region_id, name, slug)
        SELECT id, name, slug FROM regions
        ON CONFLICT (name, region_id) DO NOTHING
        """
    )
    print(f"placeholder cities created: {cur.rowcount}")
conn.commit()

placeholder cities created: 699


## 4. Localities

Legacy Locality has no slug (generated) and no lat/lng (left NULL — manual). Attached to its region's placeholder city. Localities without a region go to manual review.

In [18]:
conn.rollback()  # clear any aborted transaction from a previous failed run

localities = fetch_all("/api/localities", {"populate": "region"})
print(f"fetched {len(localities)} localities")

orphan_localities = []
with conn.cursor() as cur:
    for l in localities:
        a = attrs(l)
        name = (a.get("name") or "").strip()
        region = rel(a.get("region"))
        if not name or not region:
            orphan_localities.append((l["id"], name))
            continue
        # the region's placeholder city shares the region's name
        cur.execute(
            """
            INSERT INTO localities (city_id, name, slug)
            SELECT c.id, %s, %s
            FROM cities c JOIN regions r ON c.region_id = r.id
            WHERE r.name = %s AND c.name = r.name
            ON CONFLICT (name, city_id) DO UPDATE SET slug = EXCLUDED.slug
            """,
            (name, slugify(name), (region.get("name") or "").strip()),
        )
conn.commit()
print(f"upserted localities; MANUAL REVIEW (no region): {orphan_localities}")


fetched 5 localities
upserted localities; MANUAL REVIEW (no region): []


## 5. Verify

In [19]:
with conn.cursor() as cur:
    for t in ("countries", "regions", "cities", "localities"):
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        print(f"{t:12}: {cur.fetchone()[0]}")
conn.close()

countries   : 268
regions     : 699
cities      : 699
localities  : 5
